# DIFUSCO — Modélisation Mathématique

Ce notebook présente la formulation mathématique complète du modèle DIFUSCO appliqué au problème **TSPTW-D**  
(Voyageur de Commerce avec Fenêtres Temporelles et Disruptions).

| Section | Contenu |
|---------|--------|
| 1 | Formulation du problème — TSP comme étiquetage binaire d'arêtes |
| 2 | Processus de diffusion forward — noyau Gaussien $q(y_t \mid y_{t-1})$ |
| 3 | Processus inverse — score paramétrisé par GNN $p_\theta(y_{t-1} \mid y_t, x)$ |
| 4 | Objectif d'entraînement — borne inférieure variationnelle et perte simplifiée |
| 5 | Extension TSPTW-D — fenêtres temporelles et perturbations dans l'entrée GNN |

---
## Section 1 — Formulation du problème

### 1.1 Graphe complet

Soit un ensemble de $n$ nœuds (villes) avec coordonnées $\mathbf{x} = \{\mathbf{x}_i\}_{i=1}^n$, $\mathbf{x}_i \in \mathbb{R}^2$.

On construit le graphe complet $\mathcal{G} = (\mathcal{V}, \mathcal{E})$ avec :
- $\mathcal{V} = \{1, \ldots, n\}$ — ensemble des nœuds
- $\mathcal{E} = \mathcal{V} \times \mathcal{V}$ — toutes les paires ordonnées

La distance euclidienne entre deux nœuds :
$$d_{ij} = \|\mathbf{x}_i - \mathbf{x}_j\|_2$$

### 1.2 Tour comme matrice d'adjacence binaire

Un tour hamiltonien est représenté comme une matrice binaire $\mathbf{y}_0 \in \{0, 1\}^{n \times n}$ où :
$$y_{0,ij} = 1 \iff \text{l'arête } (i \to j) \text{ appartient au tour}$$

**Contraintes du tour** (pour que $\mathbf{y}_0$ représente un hamiltonien valide) :
$$\sum_{j=1}^n y_{0,ij} = 1 \quad \forall i \quad \text{(chaque nœud a exactement un successeur)}$$
$$\sum_{i=1}^n y_{0,ij} = 1 \quad \forall j \quad \text{(chaque nœud a exactement un prédécesseur)}$$
$$y_{0,ii} = 0 \quad \forall i \quad \text{(pas d'auto-boucle)}$$

**Remarque :** $\mathbf{y}_0$ est une matrice de permutation avec la contrainte de connexité du cycle.

### 1.3 Objectif d'optimisation

**TSP standard :**
$$\min_{\mathbf{y} \in \mathcal{H}_n} \sum_{i=1}^n \sum_{j=1}^n d_{ij} \cdot y_{ij}$$

où $\mathcal{H}_n$ est l'ensemble des matrices d'adjacence de tours hamiltoniens valides.

**TSPTW-D** ajoute des contraintes de fenêtres temporelles et des perturbations — détaillé en Section 5.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Exemple: tour sur 5 nœuds = 0→2→4→1→3→0
n = 5
tour = [0, 2, 4, 1, 3]

Y = np.zeros((n, n), dtype=int)
for k in range(n):
    i, j = tour[k], tour[(k+1) % n]
    Y[i, j] = 1

np.random.seed(42)
coords = np.random.rand(n, 2)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Matrice binaire
ax = axes[0]
im = ax.imshow(Y, cmap='Blues', vmin=0, vmax=1)
for i in range(n):
    for j in range(n):
        ax.text(j, i, str(Y[i,j]), ha='center', va='center',
                color='white' if Y[i,j] else 'gray', fontsize=12, fontweight='bold')
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels([f'j={k}' for k in range(n)])
ax.set_yticklabels([f'i={k}' for k in range(n)])
ax.set_title(r'$\mathbf{y}_0 \in \{0,1\}^{n \times n}$  (tour: 0→2→4→1→3→0)', fontsize=10)
ax.set_xlabel('successeur j')
ax.set_ylabel('prédécesseur i')

# Tour sur graphe
ax = axes[1]
for k in range(n):
    i, j = tour[k], tour[(k+1) % n]
    dx, dy = coords[j,0]-coords[i,0], coords[j,1]-coords[i,1]
    ax.annotate('', xy=coords[j], xytext=coords[i],
                arrowprops=dict(arrowstyle='->', color='steelblue', lw=1.8))
ax.scatter(coords[:,0], coords[:,1], s=120, c='white', edgecolors='black', zorder=5)
for i, (x, y) in enumerate(coords):
    ax.text(x, y, str(i), ha='center', va='center', fontsize=9, zorder=6)
ax.set_title('Tour hamiltonien correspondant', fontsize=10)
ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Représentation binaire d\'un tour TSP', fontweight='bold')
plt.tight_layout()
plt.savefig('figures/math/tour_binary.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Contrainte entrée  (somme lignes)  : {Y.sum(axis=1)} — doit être [1,1,1,1,1]')
print(f'Contrainte sortie  (somme colonnes): {Y.sum(axis=0)} — doit être [1,1,1,1,1]')
print(f'Diagonale (auto-boucles)           : {np.diag(Y)} — doit être [0,0,0,0,0]')

---
## Section 2 — Processus de diffusion forward

### 2.1 Relaxation continue

Les matrices binaires $\mathbf{y}_0 \in \{0,1\}^{n \times n}$ ne sont pas différentiables.  
DIFUSCO utilise une **relaxation Gaussienne continue** : on traite chaque $y_{0,ij} \in \{0,1\}$ comme un signal réel et on lui ajoute du bruit Gaussien progressivement.

### 2.2 Schedule de bruit

On définit un schedule de variance $\{\beta_t\}_{t=1}^T$ avec $\beta_t \in (0, 1)$ croissant :
$$\beta_t = \beta_{\min} + \frac{t-1}{T-1}(\beta_{\max} - \beta_{\min})$$

avec typiquement $\beta_{\min} = 10^{-4}$, $\beta_{\max} = 0.02$, $T = 1000$.

On dérive les quantités cumulées :
$$\alpha_t = 1 - \beta_t, \qquad \bar{\alpha}_t = \prod_{s=1}^t \alpha_s$$

$\bar{\alpha}_t$ représente la **fraction du signal original qui survit** jusqu'au pas $t$.

### 2.3 Noyau de diffusion — un pas

Le processus de Markov forward ajoute du bruit pas à pas :
$$q(\mathbf{y}_t \mid \mathbf{y}_{t-1}) = \mathcal{N}\!\left(\mathbf{y}_t;\; \sqrt{\alpha_t}\,\mathbf{y}_{t-1},\; (1-\alpha_t)\,\mathbf{I}\right)$$

Interprétation : on multiplie le signal par $\sqrt{\alpha_t} < 1$ (atténuation) et on ajoute un bruit de variance $1-\alpha_t$.

### 2.4 Astuce de reparamétrisation (forward en un seul pas)

Grâce à la propriété des Gaussiennes, on peut **sauter directement de $y_0$ à $y_t$** sans calculer tous les pas intermédiaires :

$$\boxed{q(\mathbf{y}_t \mid \mathbf{y}_0) = \mathcal{N}\!\left(\mathbf{y}_t;\; \sqrt{\bar{\alpha}_t}\,\mathbf{y}_0,\; (1-\bar{\alpha}_t)\,\mathbf{I}\right)}$$

Ce qui donne l'échantillonnage direct :
$$\mathbf{y}_t = \sqrt{\bar{\alpha}_t}\,\mathbf{y}_0 + \sqrt{1-\bar{\alpha}_t}\,\boldsymbol{\varepsilon}, \quad \boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

**Démonstration** (récurrence) :
$$q(\mathbf{y}_t \mid \mathbf{y}_0) = \int q(\mathbf{y}_t \mid \mathbf{y}_{t-1})\, q(\mathbf{y}_{t-1} \mid \mathbf{y}_0)\, d\mathbf{y}_{t-1}$$
La composition de deux Gaussiennes donne une Gaussienne avec :
$$\mu = \sqrt{\alpha_t} \cdot \sqrt{\bar{\alpha}_{t-1}}\,\mathbf{y}_0 = \sqrt{\alpha_t \bar{\alpha}_{t-1}}\,\mathbf{y}_0 = \sqrt{\bar{\alpha}_t}\,\mathbf{y}_0$$
$$\sigma^2 = \alpha_t(1-\bar{\alpha}_{t-1}) + (1-\alpha_t) = 1 - \alpha_t\bar{\alpha}_{t-1} = 1 - \bar{\alpha}_t$$

### 2.5 Comportement asymptotique

- **$t = 0$ :** $\bar{\alpha}_0 = 1 \Rightarrow \mathbf{y}_0 = \mathbf{y}_0$ (pas de bruit)
- **$t = T$ :** $\bar{\alpha}_T \approx 0 \Rightarrow \mathbf{y}_T \approx \boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ (bruit pur)

In [ ]:
import torch

T = 1000
beta_min, beta_max = 1e-4, 0.02
t_vals = torch.arange(1, T+1, dtype=torch.float32)
betas  = beta_min + (t_vals - 1) / (T - 1) * (beta_max - beta_min)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

# Forward diffusion sur le tour de l'exemple précédent
y0 = torch.tensor(Y, dtype=torch.float32)
timesteps_viz = [0, 100, 250, 500, 750, 1000]

torch.manual_seed(0)
fig, axes = plt.subplots(1, len(timesteps_viz), figsize=(14, 2.5))

for ax, t in zip(axes, timesteps_viz):
    if t == 0:
        yt = y0
    else:
        ab = alpha_bars[t-1]
        eps = torch.randn_like(y0)
        yt = ab.sqrt() * y0 + (1 - ab).sqrt() * eps
    ax.imshow(yt.numpy(), cmap='RdBu_r', vmin=-2, vmax=2)
    signal_pct = 0.0 if t == 0 else alpha_bars[t-1].item() * 100
    ax.set_title(f't={t}\n$\\bar{{\\alpha}}_t$={alpha_bars[t-1].item():.3f}' if t > 0 else 't=0\n$y_0$ (cible)',
                 fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle(r'Diffusion forward: $\mathbf{y}_t = \sqrt{\bar{\alpha}_t}\,\mathbf{y}_0 + \sqrt{1-\bar{\alpha}_t}\,\varepsilon$',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/math/forward_diffusion.png', dpi=150, bbox_inches='tight')
plt.show()

# Courbe du schedule
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
t_arr = np.arange(T)
axes[0].plot(betas.numpy(), color='tomato', lw=1.5); axes[0].set_title(r'$\beta_t$ (schedule linéaire)')
axes[1].plot(alphas.numpy(), color='steelblue', lw=1.5); axes[1].set_title(r'$\alpha_t = 1 - \beta_t$')
axes[2].plot(alpha_bars.numpy(), color='seagreen', lw=1.5)
idx50 = (alpha_bars < 0.5).nonzero(as_tuple=True)[0][0].item()
axes[2].axvline(idx50, color='orange', ls='--', lw=1, label=f't={idx50} (50% signal)')
axes[2].axhline(0.5, color='orange', ls=':', lw=0.8)
axes[2].legend(fontsize=8)
axes[2].set_title(r'$\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$')
for ax in axes:
    ax.set_xlabel('t'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('figures/math/noise_schedule.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Signal restant à t=500: {alpha_bars[499].item():.4f} ({alpha_bars[499].item()*100:.1f}%)')
print(f'Signal restant à t={T}: {alpha_bars[-1].item():.6f}')

---
## Section 3 — Processus inverse (reverse process)

### 3.1 Processus de Markov inverse

Le processus inverse tente de reconstruire $\mathbf{y}_0$ à partir de $\mathbf{y}_T \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ en appliquant successivement des pas de débruitage :

$$p_\theta(\mathbf{y}_{0:T}) = p(\mathbf{y}_T) \prod_{t=1}^T p_\theta(\mathbf{y}_{t-1} \mid \mathbf{y}_t, \mathbf{x})$$

où $\mathbf{x}$ représente les **features** du graphe (coordonnées, fenêtres temporelles, etc.).

### 3.2 Posterior forward (analytique)

La clé du DDPM est que le **posterior forward** $q(\mathbf{y}_{t-1} \mid \mathbf{y}_t, \mathbf{y}_0)$ est **analytiquement calculable** (Gaussienne) :

$$q(\mathbf{y}_{t-1} \mid \mathbf{y}_t, \mathbf{y}_0) = \mathcal{N}\!\left(\mathbf{y}_{t-1};\; \tilde{\boldsymbol{\mu}}_t,\; \tilde{\beta}_t \mathbf{I}\right)$$

avec :
$$\tilde{\boldsymbol{\mu}}_t = \underbrace{\frac{\sqrt{\bar{\alpha}_{t-1}}\,\beta_t}{1-\bar{\alpha}_t}}_{c_1}\,\mathbf{y}_0 + \underbrace{\frac{\sqrt{\alpha_t}(1-\bar{\alpha}_{t-1})}{1-\bar{\alpha}_t}}_{c_2}\,\mathbf{y}_t$$

$$\tilde{\beta}_t = \frac{(1-\bar{\alpha}_{t-1})\,\beta_t}{1-\bar{\alpha}_t}$$

**Démonstration** (règle de Bayes + propriétés Gaussiennes) :
$$q(\mathbf{y}_{t-1} \mid \mathbf{y}_t, \mathbf{y}_0) \propto q(\mathbf{y}_t \mid \mathbf{y}_{t-1})\, q(\mathbf{y}_{t-1} \mid \mathbf{y}_0)$$

En développant les densités Gaussiennes et en complétant le carré, on obtient les formules ci-dessus.

### 3.3 Score paramétrisé par GNN

Le réseau $f_\theta$ prédit **directement $\mathbf{y}_0$** (plutôt que le bruit $\boldsymbol{\varepsilon}$, comme certains travaux) :

$$\hat{\mathbf{y}}_0 = f_\theta(\mathbf{y}_t, t, \mathbf{x}) \in [0, 1]^{n \times n}$$

Le processus inverse **DDPM** devient alors :

$$\boxed{p_\theta(\mathbf{y}_{t-1} \mid \mathbf{y}_t, \mathbf{x}) = \mathcal{N}\!\left(\mathbf{y}_{t-1};\; c_1 \hat{\mathbf{y}}_0 + c_2 \mathbf{y}_t,\; \tilde{\beta}_t \mathbf{I}\right)}$$

En pratique on échantillonne :
$$\mathbf{y}_{t-1} = c_1 \hat{\mathbf{y}}_0 + c_2 \mathbf{y}_t + \sqrt{\tilde{\beta}_t}\,\boldsymbol{\varepsilon}, \quad \boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

### 3.4 DDIM — Reverse déterministe accéléré

DDIM (Song et al., 2020) reformule le reverse comme un **processus déterministe** :

$$\boxed{\mathbf{y}_{t-1} = \sqrt{\bar{\alpha}_{t-1}}\,\hat{\mathbf{y}}_0 + \sqrt{1-\bar{\alpha}_{t-1}}\underbrace{\frac{\mathbf{y}_t - \sqrt{\bar{\alpha}_t}\,\hat{\mathbf{y}}_0}{\sqrt{1-\bar{\alpha}_t}}}_{\text{direction de bruit}}}$$

Avantages DDIM :
- **50 pas** suffisent au lieu de 1000 (20× plus rapide)
- Déterministe → reproductible pour un même $\mathbf{y}_T$
- Qualité comparable ou supérieure

| | DDPM | DDIM |
|---|---|---|
| Pas nécessaires | $T = 1000$ | $S = 50$ |
| Stochastique | ✓ | ✗ |
| Vitesse | lent | rapide |
| Paramètre $\eta$ | — | 0 (déterministe) |

In [ ]:
# Illustration: coefficients c1, c2, beta_tilde au cours du reverse
ab = alpha_bars.numpy()
ab_prev = np.concatenate([[1.0], ab[:-1]])  # alpha_bar_{t-1}, avec alpha_bar_0 = 1
betas_np = betas.numpy()
alphas_np = alphas.numpy()

c1 = np.sqrt(ab_prev) * betas_np / (1 - ab)
c2 = np.sqrt(alphas_np) * (1 - ab_prev) / (1 - ab)
beta_tilde = (1 - ab_prev) * betas_np / (1 - ab)

fig, axes = plt.subplots(1, 3, figsize=(13, 3))

axes[0].plot(c1, color='steelblue', lw=1.2)
axes[0].set_title(r'$c_1$ — poids de $\hat{\mathbf{y}}_0$', fontsize=9)
axes[0].set_xlabel('t (reverse: T→1)')

axes[1].plot(c2, color='tomato', lw=1.2)
axes[1].set_title(r'$c_2$ — poids de $\mathbf{y}_t$', fontsize=9)
axes[1].set_xlabel('t (reverse: T→1)')

axes[2].plot(beta_tilde, color='seagreen', lw=1.2)
axes[2].set_title(r'$\tilde{\beta}_t$ — variance du posterior', fontsize=9)
axes[2].set_xlabel('t (reverse: T→1)')

for ax in axes:
    ax.grid(alpha=0.3)

plt.suptitle('Coefficients DDPM du processus inverse', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.savefig('figures/math/reverse_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

# Montrer que c1 + c2 ≠ 1 en général (pas une interpolation barycentrique simple)
print(f'c1 + c2 à t=500: {c1[499] + c2[499]:.4f}  (≈ 1 car signal ≈ bruit)')
print(f'c1 à t=1 (dernier pas): {c1[0]:.4f}  — ≈ sqrt(alpha_bar_0)*beta_1/(1-alpha_bar_1)')
print(f'beta_tilde à t=1: {beta_tilde[0]:.8f}  — petit: on est quasi-certain de y0')

---
## Section 4 — Objectif d'entraînement

### 4.1 Borne inférieure variationnelle (ELBO)

On maximise la log-vraisemblance $\log p_\theta(\mathbf{y}_0 \mid \mathbf{x})$ par un lower bound (ELBO) :

$$\mathcal{L} = \mathbb{E}_q\left[-\log p_\theta(\mathbf{y}_0 \mid \mathbf{y}_1)\right] + \sum_{t=2}^T \mathbb{E}_q\left[D_{\mathrm{KL}}\!\left(q(\mathbf{y}_{t-1} \mid \mathbf{y}_t, \mathbf{y}_0) \,\|\, p_\theta(\mathbf{y}_{t-1} \mid \mathbf{y}_t, \mathbf{x})\right)\right] + D_{\mathrm{KL}}\!\left(q(\mathbf{y}_T \mid \mathbf{y}_0) \,\|\, p(\mathbf{y}_T)\right)$$

Le dernier terme est constant (non dépendant de $\theta$) et peut être ignoré.

### 4.2 Perte simplifiée (Ho et al., 2020)

Puisque les deux distributions de l'argument KL sont Gaussiennes (et donc les termes KL se réduisent à des MSE sur les moyennes), Ho et al. montrent que l'on peut simplifier à :

$$\mathcal{L}_{\text{simple}} = \mathbb{E}_{t, \mathbf{y}_0, \boldsymbol{\varepsilon}} \left[\left\|\mathbf{y}_0 - f_\theta(\mathbf{y}_t, t, \mathbf{x})\right\|^2\right]$$

C'est une **perte MSE** entre la cible propre $\mathbf{y}_0$ et la prédiction $\hat{\mathbf{y}}_0$.

### 4.3 Perte BCE (choix de DIFUSCO)

Puisque $y_{0,ij} \in \{0, 1\}$, on traite la prédiction comme un problème de **classification binaire** et on utilise la **Binary Cross-Entropy** :

$$\boxed{\mathcal{L}_{\text{BCE}} = -\mathbb{E}_{t, (i,j)} \left[y_{0,ij} \log \hat{y}_{0,ij} + (1 - y_{0,ij}) \log(1 - \hat{y}_{0,ij})\right]}$$

Avantages de la BCE :
- Adaptée à des cibles binaires $\{0,1\}$
- Gradient fort quand la prédiction est certaine mais fausse
- Naturellement bornée : $\hat{y}_{0,ij} \in (0,1)$ via sigmoid

**Comparaison MSE vs BCE :**

| Critère | MSE | BCE |
|---------|-----|-----|
| Cible binaire | sous-optimal | adapté |
| Gradient à la limite | faible | fort |
| Prédiction | $\hat{y} \in \mathbb{R}$ | $\hat{y} \in (0,1)$ (sigmoid) |
| Interprétation | distance L2 | probabilité log-vraisemblance |

### 4.4 Algorithme d'entraînement complet

```
Répéter jusqu'à convergence :
  1. Échantillonner (coords, y_0, x) depuis le dataset
  2. Échantillonner t ~ Uniforme({1, ..., T})
  3. Échantillonner ε ~ N(0, I)
  4. Calculer y_t = sqrt(ᾱ_t) · y_0 + sqrt(1 - ᾱ_t) · ε
  5. Prédire ŷ_0 = sigmoid(f_θ(y_t, t, x))
  6. Calculer L = BCE(y_0, ŷ_0)
  7. Mettre à jour θ par descente de gradient ∇_θ L
```

In [ ]:
# Comparaison MSE vs BCE pour des cibles binaires
y0_true = 1.0  # cible: arête présente
y_pred_range = np.linspace(0.01, 0.99, 200)

mse  = (y0_true - y_pred_range) ** 2
bce  = -(y0_true * np.log(y_pred_range) + (1 - y0_true) * np.log(1 - y_pred_range))

# Gradients
grad_mse = -2 * (y0_true - y_pred_range)
grad_bce = -(y0_true / y_pred_range - (1 - y0_true) / (1 - y_pred_range))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(y_pred_range, mse,  label='MSE',  color='tomato',    lw=2)
axes[0].plot(y_pred_range, bce,  label='BCE',  color='steelblue', lw=2)
axes[0].axvline(1.0, color='green', ls='--', lw=1, label='y₀=1 (cible)')
axes[0].set_xlabel(r'$\hat{y}_{0,ij}$ (prédiction)')
axes[0].set_ylabel('Perte')
axes[0].set_title('Perte MSE vs BCE (cible = 1)', fontsize=10)
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(y_pred_range, np.abs(grad_mse), label='|grad MSE|', color='tomato',    lw=2)
axes[1].plot(y_pred_range, np.abs(grad_bce), label='|grad BCE|', color='steelblue', lw=2)
axes[1].axvline(1.0, color='green', ls='--', lw=1)
axes[1].set_xlabel(r'$\hat{y}_{0,ij}$ (prédiction)')
axes[1].set_ylabel('|gradient|')
axes[1].set_title('Magnitude du gradient', fontsize=10)
axes[1].set_ylim(0, 10)
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Comparaison des fonctions de perte pour cibles binaires', fontweight='bold')
plt.tight_layout()
plt.savefig('figures/math/loss_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('BCE à ŷ=0.1 (prédiction fausse) :', f'{-(np.log(0.1)):.3f}')
print('MSE à ŷ=0.1 (prédiction fausse) :', f'{(1.0-0.1)**2:.3f}')
print('→ BCE punit beaucoup plus sévèrement les prédictions erronées avec forte confiance')

---
## Section 5 — Extension TSPTW-D

### 5.1 Définition du problème TSPTW-D

Soit $n$ clients et un dépôt (nœud $0$). Pour chaque client $i \in \{1,\ldots,n\}$ :
- **Coordonnées** : $\mathbf{x}_i = (x_i, y_i) \in [0,1]^2$
- **Fenêtre temporelle** : $[a_i, b_i]$ — le service doit commencer dans cet intervalle
- **Durée de service** : $s_i \geq 0$ — temps passé chez le client
- **Perturbations** : ensemble $\mathcal{P}_i$ de paires $(\delta_{ij}^{(\text{depart})}, \delta_{ij}^{(\text{arrive})})$ représentant des disruptions de trajet

**Temps de trajet effectif** entre $i$ et $j$ avec perturbation $\delta$ :
$$\tau_{ij} = d_{ij} + \delta_{ij}$$

**Contrainte de fenêtre temporelle** (si on arrive en $i$ au temps $T_i^{\text{arr}}$) :
$$T_i^{\text{début}} = \max(T_i^{\text{arr}}, a_i) \leq b_i$$

**Objectif TSPTW-D** :
$$\min_{\pi \in \mathcal{H}_n} \left[ \underbrace{\sum_{k=0}^{n-1} d_{\pi(k), \pi(k+1)}}_{\text{distance totale}} + \lambda \underbrace{\sum_{i=1}^n \max\left(0,\, T_i^{\text{arr}} - b_i\right)}_{\text{pénalité TW}} \right]$$

### 5.2 Encodage des features nœuds

Chaque nœud $i$ est encodé avec 5 features normalisées :

$$\phi_i^{\text{node}} = \left(x_i,\; y_i,\; \frac{a_i}{T_{\max}},\; \frac{b_i}{T_{\max}},\; \frac{s_i}{T_{\max}}\right) \in [0,1]^5$$

où $T_{\max} = 200$ minutes (horizon temporel normalisateur).

**Dépôt (nœud 0) :** les features TW et service sont mis à 0.

### 5.3 Encodage des features arêtes

Pour chaque arête $(i,j)$, on calcule le **pire cas de temps de trajet** face aux perturbations connues :

$$\phi_{ij}^{\text{edge}} = \max_{\delta \in \mathcal{P}_{ij}} (d_{ij} + \delta) = d_{ij} + \max_{\delta \in \mathcal{P}_{ij}} \delta$$

Cette feature alerte le modèle sur les arêtes potentiellement dangereuses (routes perturbées).

### 5.4 Entrée complète du GNN

Au pas $t$ du reverse, l'entrée du réseau $f_\theta$ est :

$$f_\theta\!\left(\underbrace{\phi^{\text{node}} \in \mathbb{R}^{n \times 5}}_{\text{features nœuds}},\; \underbrace{\mathbf{y}_t \in \mathbb{R}^{n \times n}}_{\text{matrice bruitée}},\; \underbrace{t \in \{1,\ldots,T\}}_{\text{pas temporel}},\; \underbrace{\phi^{\text{edge}} \in \mathbb{R}^{n \times n \times 1}}_{\text{features arêtes}}\right)$$

La feature d'arête d'entrée pour le GNN est la concaténation :
$$e_{ij}^{\text{in}} = \left[\underbrace{\phi_{ij}^{\text{edge}}}_{\text{pire cas TW}},\; \underbrace{y_{t,ij}}_{\text{bruit diffusion}}\right] \in \mathbb{R}^2$$

Le timestep $t$ est injecté via un embedding sinusoïdal :
$$\text{TimeEmb}(t)_k = \begin{cases} \sin\left(t / 10000^{2k/d}\right) & k \text{ pair} \\ \cos\left(t / 10000^{(2k-1)/d}\right) & k \text{ impair} \end{cases}$$

### 5.5 Architecture GNN (DifuscoGNNLayer)

Chaque couche GNN de DIFUSCO suit l'architecture de Joshi et al. (2019) avec conditionnement temporel :

**Mise à jour des arêtes** :
$$e_{ij}^{(l+1)} = \text{BN}\left(\sigma\left(W_e^{(l)} [h_i^{(l)} \| h_j^{(l)} \| e_{ij}^{(l)} \| \underbrace{W_t^{(l)}\,\text{TimeEmb}(t)}_{\text{conditioning temporel}}]\right)\right)$$

**Attention sur les arêtes** :
$$a_{ij}^{(l)} = \text{softmax}_j\left(e_{ij}^{(l+1)}\right), \quad h_i^{(l+1)} = \text{BN}\left(\sigma\left(W_n^{(l)} \sum_j a_{ij}^{(l)} \odot e_{ij}^{(l+1)}\right)\right)$$

**Sortie** (après $L$ couches) :
$$\hat{y}_{0,ij} = \sigma_{\text{sigmoid}}\left(W_o\, e_{ij}^{(L)}\right) \in (0, 1)$$

### 5.6 Récapitulatif des dimensions

| Taille | $d$ | $L$ | $T$ | Paramètres |
|--------|-----|-----|-----|------------|
| small  | 64  | 4   | 500 | ~119k |
| medium | 128 | 6   | 1000 | ~669k |
| large  | 256 | 8   | 1000 | ~2.9M |

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from data import load_dataset

# Charger une instance TSPTW-D et afficher les features
ds = load_dataset(10)
coords = ds['coords']      # (11, 2) — dépôt + 10 clients
tw     = ds['tw']          # (11, 2) — fenêtres temporelles normalisées
svc    = ds['svc']         # (11,)   — durées de service normalisées
nf     = ds['node_feats']  # (11, 5) — features nœuds
ef     = ds['edge_feats']  # (11, 11, 1) — features arêtes

n = coords.shape[0]
print(f'Nœuds: {n} (1 dépôt + {n-1} clients)')
print(f'node_feats shape: {nf.shape}  — [x, y, a/T, b/T, s/T]')
print(f'edge_feats shape: {ef.shape}  — [pire cas τ_ij]')
print()
print('Exemple nœud 1 (premier client):')
print(f'  coords     = ({coords[1,0]:.3f}, {coords[1,1]:.3f})')
print(f'  TW normalisée = [{tw[1,0]:.3f}, {tw[1,1]:.3f}]')
print(f'  svc normalisé = {svc[1]:.3f}')
print(f'  node_feats[1] = {nf[1].numpy().round(3)}')

# Visualisation des features
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Carte des nœuds avec TW
ax = axes[0]
sc = ax.scatter(coords[1:,0], coords[1:,1],
                c=(tw[1:,1]-tw[1:,0]),  # largeur de la fenêtre
                cmap='RdYlGn', s=100, edgecolors='black', zorder=5)
ax.scatter(coords[0,0], coords[0,1], marker='*', c='gold', s=300, edgecolors='black', zorder=6)
for i in range(1, n):
    ax.text(coords[i,0]+0.01, coords[i,1]+0.01, str(i), fontsize=7)
plt.colorbar(sc, ax=ax, label='Largeur TW (normalisée)')
ax.set_title('Carte clients: largeur fenêtre temporelle\n(vert=large, rouge=étroite)', fontsize=9)
ax.set_aspect('equal')

# Heatmap des features arêtes (pire cas τ)
ax = axes[1]
im = ax.imshow(ef[:,:,0].numpy(), cmap='YlOrRd')
plt.colorbar(im, ax=ax, label='pire cas τ_ij')
ax.set_title('Heatmap arêtes: pire cas τ_ij\n(rouge = route perturbée)', fontsize=9)
ax.set_xlabel('j (destination)'); ax.set_ylabel('i (origine)')

# Embedding sinusoïdal du timestep
d = 64
timesteps_sel = [1, 50, 100, 250, 500, 750, 900, 1000]
emb_matrix = np.zeros((len(timesteps_sel), d))
for k, t in enumerate(timesteps_sel):
    for dim in range(d):
        if dim % 2 == 0:
            emb_matrix[k, dim] = np.sin(t / 10000 ** (dim / d))
        else:
            emb_matrix[k, dim] = np.cos(t / 10000 ** ((dim-1) / d))

ax = axes[2]
im2 = ax.imshow(emb_matrix, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im2, ax=ax)
ax.set_yticks(range(len(timesteps_sel)))
ax.set_yticklabels([f't={t}' for t in timesteps_sel], fontsize=8)
ax.set_xlabel('dimension de l\'embedding')
ax.set_title('Embedding sinusoïdal TimeEmb(t)\n(chaque ligne = empreinte unique du pas t)', fontsize=9)

plt.suptitle('Features d\'entrée DIFUSCO pour TSPTW-D', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.savefig('figures/math/tsptwd_features.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 6 — Pipeline complet

### Diagramme récapitulatif

```
╔══════════════════════════════════════════════════════════════════════════╗
║                    DIFUSCO — Pipeline complet TSPTW-D                   ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  ENTRAÎNEMENT                                                            ║
║  ┌────────────┐   y_0 ∈{0,1}ⁿˣⁿ      ┌────────────────────────────┐   ║
║  │  Dataset   │──(tour labels)──────>  │  Forward diffusion          │   ║
║  │ TSPTW-D   │                        │  y_t = √ᾱ_t·y_0 + √(1-ᾱ_t)ε│  ║
║  └────────────┘                        └───────────┬────────────────┘   ║
║                                                    │ y_t, t             ║
║  ┌──────────────────────────────────────────────────▼──────────────┐    ║
║  │  f_θ (DifuscoModel)                                              │    ║
║  │  Entrée: [node_feats(n×5), y_t(n×n), t, edge_feats(n×n×1)]      │    ║
║  │  Architecture: L couches GNN + time conditioning                  │    ║
║  │  Sortie: ŷ_0 ∈ (0,1)ⁿˣⁿ  (prédiction de y_0 propre)             │    ║
║  └─────────────────────────────┬────────────────────────────────────┘    ║
║                                │                                         ║
║                       L = BCE(y_0, ŷ_0)                                 ║
║                       ∇_θ L → Adam update                               ║
║                                                                          ║
║  INFÉRENCE (DDIM, S=50 pas)                                             ║
║  y_T ~ N(0,I)  ──>  [reverse chain]  ──>  ŷ_0  ──>  greedy decode      ║
║                                                   ──>  tour hamiltonien  ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
```

### Propriétés clés

| Propriété | DIFUSCO | GNN classique |
|-----------|---------|---------------|
| Représentation | Matrice adj. continue $\mathbf{y}_t \in \mathbb{R}^{n\times n}$ | Scores d'arêtes $p_{ij} \in [0,1]$ |
| Processus | Reverse diffusion Markovien | Forward pass unique |
| Diversité | Multiple via $\mathbf{y}_T$ aléatoire | Déterministe |
| Best-of-N | Naturel (N tirages de $\mathbf{y}_T$) | Possible mais moins naturel |
| Coût inférence | $O(S \cdot n^2)$ avec $S=50$ | $O(L \cdot n^2)$ avec $L \ll S$ |
| Conditioning | Timestep $t$ injecté dans GNN | — |

### Références

1. **DIFUSCO** : Sun & Yang, *"DIFUSCO: Graph-based Diffusion Solvers for Combinatorial Optimization"*, NeurIPS 2023
2. **DDPM** : Ho et al., *"Denoising Diffusion Probabilistic Models"*, NeurIPS 2020
3. **DDIM** : Song et al., *"Denoising Diffusion Implicit Models"*, ICLR 2021
4. **GNN-TSP** : Joshi et al., *"An Efficient Graph Convolutional Network Technique for the Travelling Salesman Problem"*, 2019